# Repository of similarity measures

This notebook includes user-defined functions for various user-similarity measures used for collaborative filtering.

## Notations
* $I$: Set of all rated items
* $U$: Set of all users
* $r^u_{i}$: User rating for item $i\in I$ by user $u\in U$
* $I^{u}$: Set of all items rated by user $u$
* $U_{i}$: Set of all users who rated item $i$
* $\bar{r}^u=\frac{\sum_{i\in I^u} r^u_{i}}{|I^u|}$: The average rating by user $u$
* $\bar{r}_i=\frac{\sum_{u\in U_i}r^u_i}{|I_i|}$: The average rating of item $i$
* $I^{uv}=I^u\cap I^v$: The set of commonly rated items by users $u, v\in U$.

Benchmark similarity measures

1. PCC $\checkmark$
2. Vector similarity (COS) $\checkmark$
3. ACOS $\checkmark$
4. AMI
5. ARI
6. SRC (Spearman Rank-order Correlation)
7. Kendall's tau
8. Jaccard index
9. Euclidean similarity formula
10. Manhattan similarity formula
11. Chebyshev similarity
12. Improved triangle similarity complemented with user rating preferences (ITR)
13. Improved PCC weighted with RPB (IPWR)

# PCC (Pearson Correlation Coefficient)
$$
PCC(u,v)=\frac{\sum_{i\in I^{uv}}(r^u_i-\bar{r}^u)(r^v_i-\bar{r}^v)}{\sqrt{\sum_{i\in I^{uv}}(r^u_i-\bar{r}^u)^2}\sqrt{\sum_{i\in I^{uv}}(r^v_i-\bar{r}^v)^2}}
$$

In [4]:
import numpy as np

def pcc(x, y):
    """
    Computes Pearson correlation coefficient (PCC) between two users x and y,
    centering by each user's mean over all rated items (not just common ones).
    """
    # User means (over their own non-NaN ratings)
    mean_x = np.nanmean(x)
    mean_y = np.nanmean(y)

    # Identify co-rated items
    mask = ~np.isnan(x) & ~np.isnan(y)

    if np.sum(mask) < 2:
        return np.nan  # Not enough common ratings

    # Centered ratings on co-rated items
    x_centered = x[mask] - mean_x
    y_centered = y[mask] - mean_y

    # Compute PCC
    numerator = np.sum(x_centered * y_centered)
    denominator = np.sqrt(np.sum(x_centered**2)) * np.sqrt(np.sum(y_centered**2))

    return numerator / denominator if denominator != 0 else np.nan

# COS (Cosine)
$$
COS(u,v)=\frac{\sum_{i\in I^{uv}}r^u_i r^v_i}{\sqrt{\sum_{i\in I^{u}} (r^u_i)^2}\sqrt{\sum_{i\in I^{v}}(r^v_i)^2}}
$$

Note that the norm are calculated not over co-items but individually rated items.

In [5]:
import numpy as np

def cos(x, y):
    """
    Compute cosine similarity between x and y:
    - Numerator: dot product over co-rated items
    - Denominator: full norm of x and y over their non-NaN entries
    """
    # Co-rated mask
    mask_common = ~np.isnan(x) & ~np.isnan(y)
    if np.sum(mask_common) == 0:
        return np.nan

    # Numerator: dot product over co-rated items
    numerator = np.nansum(x[mask_common] * y[mask_common])

    # Denominator: full norms (not just on co-rated items)
    norm_x = np.sqrt(np.nansum(x[~np.isnan(x)]**2))
    norm_y = np.sqrt(np.nansum(y[~np.isnan(y)]**2))

    denominator = norm_x * norm_y

    return numerator / denominator if denominator != 0 else np.nan


# ACOS (Adjusted Cosine)
$$
ACOS(u,v)=\frac{\sum_{i\in I^{uv}}(r^u_i-\bar{r}_i)(r^v_i-\bar{r}_i)}{\sqrt{\sum_{i\in I^{uv}}(r^u_i-\bar{r}_i)^2}\sqrt{\sum_{i\in I^{uv}}(r^v_i-\bar{r}_i)^2}}
$$

The formula looks similar to PCC. The difference is the the averages are not user-wise averages but the item-wise average.

In [6]:
def acos(x, y, item_means):
    mask = ~np.isnan(x) & ~np.isnan(y)
    if np.sum(mask) < 2: return np.nan
    x_adj = x[mask] - item_means[mask]
    y_adj = y[mask] - item_means[mask]
    numerator = np.sum(x_adj * y_adj)
    denominator = np.sqrt(np.sum(x_adj ** 2)) * np.sqrt(np.sum(y_adj ** 2))
    return numerator / denominator if denominator != 0 else np.nan

# Adjusted Mutual Information (AMI)

For a fixed user $u\in U$, consider the set $I^u$. Now, for each item $i\in I^u$, define
$$
P^u(r)=\frac{\text{The count of rating category $r$'s in $I^u$}}{|I^u|}
$$
Now, we define the entropy of user $u$ as
$$H(u) = -\sum_{i\in I^u} P^u(i)\log P^u(i).$$

For two users $u$ and $v$, define
$$
P^{u,v}(r,s)=\frac{\text{The count of rating pair $(r,s)$'s in $I^{uv}$}}{|I^{uv}|}
$$

On the other hand, the mutual information index (MI) is defined as follows:
$$
MI(u,v)=\sum_{r=1}^5\sum_{s=1}^5 P^{uv}(r,s)\log\frac{P^{uv}(r,s)}{P^u(r)P^v(s)}
$$

Finally, the adjusted mutual information (AMI) is defined as

$$
AMI(u,v)=\frac{MI(u,v)-E(MI(u,v))}{\max\{H(u),H(v)\}-E(MI(u,v))}
$$

where

$$
E(MI(u,v))=\sum_{r=1}^5\sum_{s=1}^5 P^{uv}(r,s)MI(r,s)
$$

In [7]:
import numpy as np
from sklearn.metrics import adjusted_mutual_info_score

def ami(u_ratings, v_ratings):
    """
    Computes Adjusted Mutual Information (AMI) between two users' ratings.

    Args:
        u_ratings: numpy array of ratings by user u (np.nan for missing).
        v_ratings: numpy array of ratings by user v (np.nan for missing).

    Returns:
        AMI score (float), np.nan if insufficient data.
    """
    # Identify co-rated items (both users rated the same item)
    mask = ~np.isnan(u_ratings) & ~np.isnan(v_ratings)

    if np.sum(mask) < 2:
        return np.nan  # Not enough co-rated items to compute meaningful AMI

    u_clean = u_ratings[mask].astype(int)
    v_clean = v_ratings[mask].astype(int)

    return adjusted_mutual_info_score(u_clean, v_clean)

# ✅ Example usage:
# Suppose user u and v have rated 5 items (np.nan indicates missing ratings)
u = np.array([5, 3, np.nan, 1, 4])
v = np.array([5, np.nan, 2, 1, 4])

ami_score = ami(u, v)
print("AMI:", ami_score)

AMI: 1.0


In [8]:
import numpy as np

def cpcc(x, y, max_rating):
    """
    Computes Pearson correlation coefficient (PCC) between two users x and y,
    centering by each user's mean over all rated items (not just common ones).
    """
    import statistics
    # User medians (over their own non-NaN ratings)
    median_x = (max_rating+1)/2
    median_y = (max_rating+1)/2

    # Identify co-rated items
    mask = ~np.isnan(x) & ~np.isnan(y)

    if np.sum(mask) < 2:
        return np.nan  # Not enough common ratings

    # Centered ratings on co-rated items
    x_centered = x[mask] - median_x
    y_centered = y[mask] - median_y

    # Compute PCC
    numerator = np.sum(x_centered * y_centered)
    denominator = np.sqrt(np.sum(x_centered**2)) * np.sqrt(np.sum(y_centered**2))

    return numerator / denominator if denominator != 0 else np.nan

In [9]:
import numpy as np
from sklearn.metrics import adjusted_rand_score

def ari(u_ratings, v_ratings):
    """
    Computes Adjusted Rand Index (ARI) between two users' ratings over co-rated items.

    Args:
        u_ratings: numpy array of user u's ratings (np.nan for missing).
        v_ratings: numpy array of user v's ratings (np.nan for missing).

    Returns:
        ARI score (float), np.nan if insufficient co-rated items.
    """
    # Mask for co-rated items
    mask = ~np.isnan(u_ratings) & ~np.isnan(v_ratings)

    if np.sum(mask) < 2:
        return np.nan  # Not enough co-rated items

    u_clean = u_ratings[mask].astype(int)
    v_clean = v_ratings[mask].astype(int)

    return adjusted_rand_score(u_clean, v_clean)

# ✅ Example usage:
u = np.array([5, 3, np.nan, 1, 4, 2])
v = np.array([5, np.nan, 3, 1, 4, 2])

ari_score = ari(u, v)
print("ARI:", ari_score)

ARI: 1.0


In [10]:
import numpy as np
from scipy.stats import spearmanr

def src(u_ratings, v_ratings):
    """
    Computes Spearman rank correlation between two users' ratings over co-rated items.

    Args:
        u_ratings: numpy array of user u's ratings (np.nan for missing).
        v_ratings: numpy array of user v's ratings (np.nan for missing).

    Returns:
        Spearman rank correlation coefficient (float), np.nan if insufficient data.
    """
    # Mask for co-rated items
    mask = ~np.isnan(u_ratings) & ~np.isnan(v_ratings)

    if np.sum(mask) < 2:
        return np.nan  # Not enough data to compute correlation

    u_clean = u_ratings[mask]
    v_clean = v_ratings[mask]

    corr, _ = spearmanr(u_clean, v_clean)
    return corr

# ✅ Example usage:
u = np.array([5, 3, np.nan, 1, 4, 2])
v = np.array([4, np.nan, 3, 1, 5, 2])

spearman_score = src(u, v)
print("Spearman Rank Correlation:", spearman_score)

Spearman Rank Correlation: 0.7999999999999999


In [11]:
import numpy as np
from scipy.stats import kendalltau

def kendall_tau_b(u_ratings, v_ratings):
    """
    Computes Kendall's Tau-b rank correlation between two users' ratings over co-rated items.

    Args:
        u_ratings: numpy array of user u's ratings (np.nan for missing).
        v_ratings: numpy array of user v's ratings (np.nan for missing).

    Returns:
        Kendall's Tau-b coefficient (float), np.nan if insufficient data.
    """
    # Mask for co-rated items
    mask = ~np.isnan(u_ratings) & ~np.isnan(v_ratings)

    if np.sum(mask) < 2:
        return np.nan  # Not enough co-rated items

    u_clean = u_ratings[mask]
    v_clean = v_ratings[mask]

    tau, _ = kendalltau(u_clean, v_clean, variant='b')
    return tau

# ✅ Example usage:
u = np.array([5, 3, np.nan, 1, 4, 2])
v = np.array([4, np.nan, 3, 1, 5, 2])

kendall_score = kendall_tau_b(u, v)
print("Kendall's Tau-b:", kendall_score)


Kendall's Tau-b: 0.6666666666666669


In [12]:
import numpy as np

def jaccard(x, y):
    """
    Computes Pearson correlation coefficient (PCC) between two users x and y,
    centering by each user's mean over all rated items (not just common ones).
    """

    # Identify co-rated items
    mask = ~np.isnan(x) & ~np.isnan(y)
    mask2 = ~np.isnan(x) | ~np.isnan(y)

    num = np.sum(mask)
    den = np.sum(mask2)

    if den == 0:
        return np.nan  # Not enough common ratings

    return num / den

In [13]:
import numpy as np

def euclidean_similarity(u_ratings, v_ratings):
    """
    Computes Euclidean similarity between two users:
    similarity = 1 / (1 + Euclidean distance over co-rated items)

    Args:
        u_ratings, v_ratings: numpy arrays of ratings (np.nan for missing)

    Returns:
        similarity (float), np.nan if insufficient co-rated items
    """
    # Co-rated mask
    mask = ~np.isnan(u_ratings) & ~np.isnan(v_ratings)

    if np.sum(mask) < 1:
        return np.nan  # No co-rated items

    distance = np.sqrt(np.sum((u_ratings[mask] - v_ratings[mask]) ** 2))
    similarity = 1 / (1 + distance)

    return similarity

# ✅ Example usage:
u = np.array([5, 3, np.nan, 1, 4, 2])
v = np.array([4, np.nan, 3, 1, 5, 2])

sim = euclidean_similarity(u, v)
print("Euclidean Similarity:", sim)


Euclidean Similarity: 0.4142135623730951


In [14]:
import numpy as np

def manhattan_similarity(u_ratings, v_ratings):
    """
    Computes Manhattan similarity between two users:
    similarity = 1 / (1 + Manhattan distance over co-rated items)

    Args:
        u_ratings, v_ratings: numpy arrays of ratings (np.nan for missing)

    Returns:
        similarity (float), np.nan if no co-rated items
    """
    mask = ~np.isnan(u_ratings) & ~np.isnan(v_ratings)

    if np.sum(mask) < 1:
        return np.nan  # No co-rated items

    distance = np.sum(np.abs(u_ratings[mask] - v_ratings[mask]))
    similarity = 1 / (1 + distance)

    return similarity

# ✅ Example usage:
u = np.array([5, 3, np.nan, 1, 4, 2])
v = np.array([4, np.nan, 3, 1, 5, 2])

sim = manhattan_similarity(u, v)
print("Manhattan Similarity:", sim)


Manhattan Similarity: 0.3333333333333333


In [15]:
import numpy as np

def chebyshev_similarity(u_ratings, v_ratings):
    """
    Computes Chebyshev similarity between two users:
    similarity = 1 / (1 + Chebyshev distance over co-rated items)

    Args:
        u_ratings, v_ratings: numpy arrays of ratings (np.nan for missing)

    Returns:
        similarity (float), np.nan if no co-rated items
    """
    mask = ~np.isnan(u_ratings) & ~np.isnan(v_ratings)

    if np.sum(mask) < 1:
        return np.nan  # No co-rated items

    distance = np.max(np.abs(u_ratings[mask] - v_ratings[mask]))
    similarity = 1 / (1 + distance)

    return similarity

# ✅ Example usage:
u = np.array([5, 3, np.nan, 1, 4, 2])
v = np.array([4, np.nan, 3, 1, 5, 2])

sim = chebyshev_similarity(u, v)
print("Chebyshev Similarity:", sim)


Chebyshev Similarity: 0.5


In [16]:
import numpy as np

def triangle_similarity(x, y):
    # Find common indices
    mask = ~np.isnan(x) & ~np.isnan(y)
    if np.sum(mask) == 0:
        return np.nan

    x_c = x[mask]
    y_c = y[mask]

    norm_diff = np.linalg.norm(x_c - y_c)
    norm_sum = np.linalg.norm(x_c) + np.linalg.norm(y_c)

    return 1 - (norm_diff / norm_sum) if norm_sum != 0 else np.nan

In [17]:
import numpy as np

def urp(x, y):
    # Extract valid ratings
    x_rated = x[~np.isnan(x)]
    y_rated = y[~np.isnan(y)]

    if len(x_rated) == 0 or len(y_rated) == 0:
        return np.nan

    mu_x = np.mean(x_rated)
    mu_y = np.mean(y_rated)
    s_x = np.std(x_rated, ddof=0)  # population std
    s_y = np.std(y_rated, ddof=0)

    diff = (mu_x - mu_y) * (s_x - s_y)
    return 1 - 1 / (1 + np.exp(-diff))

In [18]:
def itr(x, y):
  return triangle_similarity(x,y)*urp(x,y)

In [19]:
import numpy as np

def rpb(x, y):
    # Extract valid ratings
    x_rated = x[~np.isnan(x)]
    y_rated = y[~np.isnan(y)]

    if len(x_rated) == 0 or len(y_rated) == 0:
        return np.nan

    mu_x = np.mean(x_rated)
    mu_y = np.mean(y_rated)
    s_x = np.std(x_rated, ddof=0)  # population std
    s_y = np.std(y_rated, ddof=0)

    product = abs(mu_x - mu_y) * abs(s_x - s_y)
    return np.cos(product)

In [20]:
import numpy as np

def ipcc(x, y, item_means):
    """
    x: np.array, ratings of user u, possibly with np.nan
    y: np.array, ratings of user v, possibly with np.nan
    item_means: np.array, mean ratings for each item
    """
    # Identify rated items
    mask_x = ~np.isnan(x)
    mask_y = ~np.isnan(y)
    mask_common = mask_x & mask_y  # items rated by both x and y

    if np.sum(mask_common) < 2:
        return np.nan

    # User means (over their own rated items)
    mean_x = np.nanmean(x)
    mean_y = np.nanmean(y)

    # Deviations for common items
    diff_x = (x[mask_common] - mean_x) - (x[mask_common] - item_means[mask_common])
    diff_y = (y[mask_common] - mean_y) - (y[mask_common] - item_means[mask_common])

    numerator = np.sum(diff_x * diff_y)

    # Full denominators over each user's rated items
    full_diff_x = (x[mask_x] - mean_x) - (x[mask_x] - item_means[mask_x])
    full_diff_y = (y[mask_y] - mean_y) - (y[mask_y] - item_means[mask_y])

    denom_x = np.sqrt(np.sum(full_diff_x ** 2))
    denom_y = np.sqrt(np.sum(full_diff_y ** 2))

    if denom_x == 0 or denom_y == 0:
        return np.nan

    return numerator / (denom_x * denom_y)

In [21]:
def ipwr(x, y, item_means):
  return rpb(x,y)*ipcc(x,y, item_means)

In [ ]:
# Assuming all the required functions (pcc, cos, acos, etc.) are already defined

import numpy as np

# ✅ Example usage:
u = np.array([5, 3, np.nan, 1, 4, 2])
v = np.array([4, np.nan, 3, 1, 5, 2])
item_means = np.array([3.5, 3.0, 2.0, 4.0, 1.5, 2.5])

# Print header
print("{:<30s} {:>10s}".format("Measure", "Value"))
print("-" * 42)

# Print aligned results
print("{:<35s} {:>10.4f}".format("PCC:", pcc(u, v)))
print("{:<35s} {:>10.4f}".format("COS:", cos(u, v)))
print("{:<35s} {:>10.4f}".format("ACOS:", acos(u, v, item_means)))
print("{:<35s} {:>10.4f}".format("AMI:", ami(u, v)))
print("{:<35s} {:>10.4f}".format("ARI:", ari(u, v)))
print("{:<35s} {:>10.4f}".format("Spearman Rank Correlation:", src(u, v)))
print("{:<35s} {:>10.4f}".format("Kendall tau_b:", kendall_tau_b(u, v)))
print("{:<35s} {:>10.4f}".format("Jaccard index:", jaccard(u, v)))
print("{:<35s} {:>10.4f}".format("Euclidean Similarity:", euclidean_similarity(u, v)))
print("{:<35s} {:>10.4f}".format("Manhattan Similarity:", manhattan_similarity(u, v)))
print("{:<35s} {:>10.4f}".format("Chebyshev Similarity:", chebyshev_similarity(u, v)))
print("{:<35s} {:>10.4f}".format("Improved Triangle Similarity:", itr(u, v)))
print("{:<35s} {:>10.4f}".format("Improved PCC weighted with RPB:", ipwr(u, v, item_means)))

Measure                             Value
------------------------------------------
PCC:                                    0.9000
COS:                                    0.8182
ACOS:                                   0.9543
AMI:                                    0.0000
ARI:                                    1.0000
Spearman Rank Correlation:              0.8000
Kendall tau_b:                          0.6667
Jaccard index:                          0.6667
Euclidean Similarity:                   0.4142
Manhattan Similarity:                   0.3333
Chebyshev Similarity:                   0.5000
Improved Triangle Similarity:           0.4479
Improved PCC weighted with RPB:         0.8885


In [24]:
import numpy as np

def wpcc(x, y, H):
    """
    Computes Pearson correlation coefficient (PCC) between two users x and y,
    centering by each user's mean over all rated items (not just common ones).
    """
    # User means (over their own non-NaN ratings)
    mean_x = np.nanmean(x)
    mean_y = np.nanmean(y)

    # Identify co-rated items
    mask = ~np.isnan(x) & ~np.isnan(y)

    if np.sum(mask) < 2:
        return np.nan  # Not enough common ratings

    # Centered ratings on co-rated items
    x_centered = x[mask] - mean_x
    y_centered = y[mask] - mean_y

    # Compute PCC
    numerator = np.sum(x_centered * y_centered)
    denominator = np.sqrt(np.sum(x_centered**2)) * np.sqrt(np.sum(y_centered**2))

    if np.sum(mask) >= H:
      return numerator / denominator if denominator != 0 else np.nan
    else:
      return numerator/denominator*np.sum(mask)/H if denominator != 0 else np.nan

In [25]:
import numpy as np

def spcc(x, y):
    """
    Computes Pearson correlation coefficient (PCC) between two users x and y,
    centering by each user's mean over all rated items (not just common ones).
    """
    # User means (over their own non-NaN ratings)
    mean_x = np.nanmean(x)
    mean_y = np.nanmean(y)

    # Identify co-rated items
    mask = ~np.isnan(x) & ~np.isnan(y)

    if np.sum(mask) < 2:
        return np.nan  # Not enough common ratings

    # Centered ratings on co-rated items
    x_centered = x[mask] - mean_x
    y_centered = y[mask] - mean_y

    # Compute PCC
    numerator = np.sum(x_centered * y_centered)
    denominator = np.sqrt(np.sum(x_centered**2)) * np.sqrt(np.sum(y_centered**2))

    return numerator / denominator * 1/(1+np.exp(-np.sum(mask)/2)) if denominator != 0 else np.nan

In [ ]:
x = np.array([1, np.nan, 2, 3, np.nan, 4])
new_x = np.nan_to_num(x, nan=0.0)
print(new_x)

[1. 0. 2. 3. 0. 4.]


In [ ]:
# import numpy as np

# def acos(x, y):
#     """
#     Computes Pearson correlation coefficient (PCC) between two users x and y,
#     centering by each user's mean over all rated items (not just common ones).
#     """
#     # User means (over their own non-NaN ratings)

#     mean_x = np.nanmean(x)
#     mean_y = np.nanmean(y)

#     # Identify co-rated items
#     mask = ~np.isnan(x) & ~np.isnan(y)
#     mask2 = ~np.isnan(x) | ~np.isnan(y)

#     x = np.nan_to_num(x, nan=0.0)
#     y = np.nan_to_num(y, nan=0.0)

#     # Identify co-rated items
#     #mask = ~np.isnan(x) & ~np.isnan(y)

#     if np.sum(mask) < 2:
#         return np.nan  # Not enough common ratings

#     # Centered ratings on co-rated items
#     x_centered = x[mask2] - mean_x
#     y_centered = y[mask2] - mean_y

#     x_centered_den = x - mean_x
#     y_centered_den = y - mean_y

#     # Compute PCC
#     numerator = np.sum(x_centered * y_centered)
#     denominator = np.sqrt(np.sum(x_centered_den**2)) * np.sqrt(np.sum(y_centered_den**2))

#     return numerator / denominator if denominator != 0 else np.nan

In [22]:
def msd(x, y):

    # Identify co-rated items
    mask = ~np.isnan(x) & ~np.isnan(y)

    if np.sum(mask) < 2:
        return np.nan  # Not enough common ratings

    return 1-np.sum((x[mask]/5 - y[mask]/5)**2)/np.sum(mask)

In [23]:
def jmsd(x, y):
  if np.isnan(jaccard(x,y)) | np.isnan(msd(x,y)):
    return np.nan
  else:
    return jaccard(x,y)*msd(x,y)

In [ ]:
def agreement(r1, r2, min, max):
  med = (min+max)/2
  print((r1 - med) * (r2 - med))
  if (r1 - med) * (r2 - med) < 0:
    return False
  else:
    return True

In [ ]:
def proximity(r1, r2, min, max):
  if agreement(r1, r2, min, max):
    d = abs(r1-r2)
  else:
    d = 2*abs(r1-r2)
  return (2*abs(max-min)+1-d)**2

In [ ]:
def impact(r1, r2, min, max):
  med = (max+min)/2
  if agreement(r1, r2, min, max):
    return (abs(r1-med)+1)*(abs(r2-med)+1)
  else:
    return 1/((abs(r1-med)+1)*(abs(r2-med)+1))

In [ ]:
def popularity(df, i, j1, j2):
  mu = np.nanmean(df.iloc[i,:])
  r1 = df.iloc[i,j1]
  r2 = df.iloc[i,j2]
  if (r1-mu)*(r2-mu)>0:
    return 1+((r1/5+r2/5)/2-mu/5)**2
  else:
    return 1

In [ ]:
def pip(j1, j2, df, min, max):
  x = np.array(df.iloc[:,j1])
  y = np.array(df.iloc[:,j2])
  # Identify co-rated items
  mask = ~np.isnan(x) & ~np.isnan(y)

  if np.sum(mask) < 2:
      return np.nan  # Not enough common ratings

  pip = 0
  for i in range(df.shape[0]):
    if mask[i]:
      print(proximity(df.iloc[i,j1],df.iloc[i,j2],min,max), impact(df.iloc[i,j1],df.iloc[i,j2],min,max), popularity(df, i, j1, j2))
      pip += proximity(df.iloc[i,j1],df.iloc[i,j2],min,max)*impact(df.iloc[i,j1],df.iloc[i,j2],min,max)*popularity(df, i, j1, j2)
  return pip

In [ ]:
import pandas as pd


# 데이터 준비 (리스트의 리스트 형태: 6행 5열)
data = [
    [4, 5, 4, 2, 4],
    [3, 3, 3 ,1, 2],
    [5,np.nan,3,np.nan,np.nan],
    [4,np.nan,4,np.nan,np.nan]
]

I=len(data)
U=len(data[0])

df = pd.DataFrame(data)

display(df)

,0,1,2,3,4
0,4,5.0,4,2.0,4.0
1,3,3.0,3,1.0,2.0
2,5,NaN,3,NaN,NaN
3,4,NaN,4,NaN,NaN


In [ ]:
pcc_mat = [[pcc(np.array(df.iloc[:,i]), np.array(df.iloc[:,j])) for j in range(U)] for i in range(U)]
cpcc_mat = [[cpcc(np.array(df.iloc[:,i]), np.array(df.iloc[:,j]), 5) for j in range(U)] for i in range(U)]
wpcc_mat = [[wpcc(np.array(df.iloc[:,i]), np.array(df.iloc[:,j]), 50) for j in range(U)] for i in range(U)]
spcc_mat = [[spcc(np.array(df.iloc[:,i]), np.array(df.iloc[:,j])) for j in range(U)] for i in range(U)]
acos_mat = [[acos(np.array(df.iloc[:,i]), np.array(df.iloc[:,j])) for j in range(U)] for i in range(U)]
cos_mat = [[cos(np.array(df.iloc[:,i]), np.array(df.iloc[:,j])) for j in range(U)] for i in range(U)]
jaccard_mat = [[jaccard(np.array(df.iloc[:,i]), np.array(df.iloc[:,j])) for j in range(U)] for i in range(U)]
msd_mat = [[msd(np.array(df.iloc[:,i]), np.array(df.iloc[:,j])) for j in range(U)] for i in range(U)]
jmsd_mat = [[jmsd(np.array(df.iloc[:,i]), np.array(df.iloc[:,j])) for j in range(U)] for i in range(U)]
pip_mat = [[pip(j1, j2, df, 1, 5) for j2 in range(U)] for j1 in range(U)]


display("PCC", pd.DataFrame(pcc_mat))
display("CPCC", pd.DataFrame(cpcc_mat))
display("WPCC", pd.DataFrame(wpcc_mat))
display("SPCC", pd.DataFrame(spcc_mat))
display("ACOS", pd.DataFrame(acos_mat))
display("COS", pd.DataFrame(cos_mat))
display("Jaccard", pd.DataFrame(jaccard_mat))
display("MSD", pd.DataFrame(msd_mat))
display("JMSD", pd.DataFrame(jmsd_mat))
display("PIP", pd.DataFrame(pip_mat))

[ True  True  True  True] [ True  True  True  True]
[ True  True False False] [ True  True  True  True]
[ True  True  True  True] [ True  True  True  True]
[ True  True False False] [ True  True  True  True]
[ True  True False False] [ True  True  True  True]
[ True  True False False] [ True  True  True  True]
[ True  True False False] [ True  True False False]
[ True  True False False] [ True  True  True  True]
[ True  True False False] [ True  True False False]
[ True  True False False] [ True  True False False]
[ True  True  True  True] [ True  True  True  True]
[ True  True False False] [ True  True  True  True]
[ True  True  True  True] [ True  True  True  True]
[ True  True False False] [ True  True  True  True]
[ True  True False False] [ True  True  True  True]
[ True  True False False] [ True  True  True  True]
[ True  True False False] [ True  True False False]
[ True  True False False] [ True  True  True  True]
[ True  True False False] [ True  True False False]
[ True  True

'PCC'

,0,1,2,3,4
0,1.000000,0.707107,0.0,0.707107,0.707107
1,0.707107,1.000000,1.0,1.000000,1.000000
2,0.000000,1.000000,1.0,1.000000,1.000000
3,0.707107,1.000000,1.0,1.000000,1.000000
4,0.707107,1.000000,1.0,1.000000,1.000000


'CPCC'

,0,1,2,3,4
0,1.000000,1.000000,0.577350,-0.447214,0.707107
1,1.000000,1.000000,1.000000,-0.447214,0.707107
2,0.577350,1.000000,1.000000,-0.447214,0.707107
3,-0.447214,-0.447214,-0.447214,1.000000,0.316228
4,0.707107,0.707107,0.707107,0.316228,1.000000


...

,0,1,2,3,4
0,1.00,0.98,0.96,0.84,0.98
1,0.98,1.00,0.98,0.74,0.96
2,0.96,0.98,1.00,0.84,0.98
3,0.84,0.74,0.84,1.00,0.90
4,0.98,0.96,0.98,0.90,1.00


'JMSD'

,0,1,2,3,4
0,1.00,0.49,0.96,0.42,0.49
1,0.49,1.00,0.49,0.74,0.96
2,0.96,0.49,1.00,0.42,0.49
3,0.42,0.74,0.42,1.00,0.90
4,0.49,0.96,0.49,0.90,1.00


'PIP'

,0,1,2,3,4
0,1488.8448,473.6928,877.6848,153.2500,452.5184
1,473.6928,853.1568,473.6928,148.5000,519.5264
2,877.6848,473.6928,814.9248,153.2500,452.5184
3,153.2500,148.5000,153.2500,1152.1440,402.6916
4,452.5184,519.5264,452.5184,402.6916,650.5920


# Toy Example — Similarity_measures.ipynb 구현 검증

이 섹션은 노트북에 정의된 함수들을 **위에서부터 순서대로 실행한 뒤** 실행해야 한다.  
동일한 5×6 토이 데이터에 대해 각 함수를 직접 호출하여 유사도 행렬을 계산한다.

In [26]:
import numpy as np
import pandas as pd

# 토이 데이터 (User × Item, 5명 × 6아이템)
toy_data = [
    [3, np.nan, 4, np.nan, np.nan, np.nan],   # U1
    [np.nan, 5, np.nan, 3, 5, 3],             # U2
    [3, 4, 4, 3, 4, 3],                       # U3
    [1, 2, 1, np.nan, 2, np.nan],             # U4
    [1, np.nan, 5, np.nan, np.nan, np.nan],   # U5
]

USER_LABELS = ['U1', 'U2', 'U3', 'U4', 'U5']
ITEM_LABELS = ['I1', 'I2', 'I3', 'I4', 'I5', 'I6']

toy_df = pd.DataFrame(toy_data, index=USER_LABELS, columns=ITEM_LABELS)
print("=== 토이 데이터 (User × Item) ===")
display(toy_df)

# 유저 벡터 리스트 (각 유저의 6개 아이템 평점 배열)
users = [np.array(toy_df.iloc[i, :]) for i in range(len(USER_LABELS))]

# 아이템 평균 (ACOS, IPCC, IPWR 계산에 필요)
item_means = np.nanmean(toy_df.values, axis=0)
print("\n아이템 평균:", item_means)

=== 토이 데이터 (User × Item) ===


,I1,I2,I3,I4,I5,I6
U1,3.0,NaN,4.0,NaN,NaN,NaN
U2,NaN,5.0,NaN,3.0,5.0,3.0
U3,3.0,4.0,4.0,3.0,4.0,3.0
U4,1.0,2.0,1.0,NaN,2.0,NaN
U5,1.0,NaN,5.0,NaN,NaN,NaN



아이템 평균: [2.         3.66666667 3.5        3.         3.66666667 3.        ]


In [27]:
import warnings
warnings.filterwarnings('ignore')

n = len(USER_LABELS)

def make_mat(func):
    """n×n 유사도 행렬 생성"""
    return [[func(users[i], users[j]) for j in range(n)] for i in range(n)]

def show_mat(name, mat):
    df_m = pd.DataFrame(mat, index=USER_LABELS, columns=USER_LABELS).round(4)
    print(f"\n{'='*50}\n  {name}\n{'='*50}")
    display(df_m)

# Similarity_measures.ipynb에 정의된 함수로 각 행렬 계산
show_mat("PCC",           make_mat(lambda u, v: pcc(u, v)))
show_mat("COS",           make_mat(lambda u, v: cos(u, v)))
show_mat("ACOS",          make_mat(lambda u, v: acos(u, v, item_means)))
show_mat("AMI",           make_mat(lambda u, v: ami(u, v)))
show_mat("ARI",           make_mat(lambda u, v: ari(u, v)))
show_mat("SRC",           make_mat(lambda u, v: src(u, v)))
show_mat("Kendall tau_b", make_mat(lambda u, v: kendall_tau_b(u, v)))
show_mat("Jaccard",       make_mat(lambda u, v: jaccard(u, v)))
show_mat("Euclidean",     make_mat(lambda u, v: euclidean_similarity(u, v)))
show_mat("Manhattan",     make_mat(lambda u, v: manhattan_similarity(u, v)))
show_mat("Chebyshev",     make_mat(lambda u, v: chebyshev_similarity(u, v)))
show_mat("MSD",           make_mat(lambda u, v: msd(u, v)))
show_mat("JMSD",          make_mat(lambda u, v: jmsd(u, v)))
show_mat("ITR",           make_mat(lambda u, v: itr(u, v)))
show_mat("IPWR",          make_mat(lambda u, v: ipwr(u, v, item_means)))
show_mat("CPCC (max=5)",  make_mat(lambda u, v: cpcc(u, v, 5)))
show_mat("WPCC (H=50)",   make_mat(lambda u, v: wpcc(u, v, 50)))
show_mat("SPCC",          make_mat(lambda u, v: spcc(u, v)))


  PCC


,U1,U2,U3,U4,U5
U1,1.0,NaN,1.0,0.0,1.0
U2,NaN,1.0,1.0,1.0,NaN
U3,1.0,1.0,1.0,0.5,1.0
U4,0.0,1.0,0.5,1.0,0.0
U5,1.0,NaN,1.0,0.0,1.0



  COS


,U1,U2,U3,U4,U5
U1,1.0000,NaN,0.5774,0.4427,0.9021
U2,NaN,1.0000,0.8122,0.7670,NaN
U3,0.5774,0.8122,1.0000,0.8398,0.5208
U4,0.4427,0.7670,0.8398,1.0000,0.3721
U5,0.9021,NaN,0.5208,0.3721,1.0000



  ACOS


...

,U1,U2,U3,U4,U5
U1,1.0000,NaN,1.0000,-0.7071,0.7071
U2,NaN,1.0,1.0000,-1.0000,NaN
U3,1.0000,1.0,1.0000,-0.7303,0.7071
U4,-0.7071,-1.0,-0.7303,1.0000,0.0000
U5,0.7071,NaN,0.7071,0.0000,1.0000



  WPCC (H=50)


,U1,U2,U3,U4,U5
U1,0.04,NaN,0.04,0.00,0.04
U2,NaN,0.08,0.08,0.04,NaN
U3,0.04,0.08,0.12,0.04,0.04
U4,0.00,0.04,0.04,0.08,0.00
U5,0.04,NaN,0.04,0.00,0.04



  SPCC


,U1,U2,U3,U4,U5
U1,0.7311,NaN,0.7311,0.0000,0.7311
U2,NaN,0.8808,0.8808,0.7311,NaN
U3,0.7311,0.8808,0.9526,0.4404,0.7311
U4,0.0000,0.7311,0.4404,0.8808,0.0000
U5,0.7311,NaN,0.7311,0.0000,0.7311
